
# 📌 Q1. How do you treat heteroscedasticity in regression?

---

## **🔹 Definition**

Heteroscedasticity means that the **variance of errors (residuals) is not constant** across all levels of the independent variable(s).
This violates one of the key assumptions of **linear regression** (constant variance of errors = *homoscedasticity*).

* When present, it can make:

  * Coefficient estimates still **unbiased**,
  * But **standard errors wrong** → leading to incorrect p-values & unreliable hypothesis tests.

---

## **🔹 Causes**

* Outliers or influential data points
* Skewed distribution of variables
* Wrong functional form (e.g., using linear when relationship is nonlinear)
* Scale differences in predictors

---

## **🔹 Detection**

* **Residual vs Fitted plot** → Funnel shape indicates heteroscedasticity
* **Breusch–Pagan test**
* **White test**
* **Goldfeld–Quandt test**

---

## **🔹 Treatment Methods**

1. **Transform the dependent variable**

   * Apply transformations like:

     * Log(y), √y, Box-Cox
   * Helps stabilize variance.

   ```python
   import numpy as np
   y_transformed = np.log(y)   # Example
   ```

2. **Weighted Least Squares (WLS)**

   * Give smaller weights to data points with higher variance.
   * Regression minimizes **weighted residuals**.

   ```python
   import statsmodels.api as sm
   wls_model = sm.WLS(y, X, weights=1/(abs(residuals))).fit()
   ```

3. **Robust Standard Errors (Heteroscedasticity-consistent SE)**

   * Use **HC standard errors** (e.g., White’s correction).
   * Coefficients remain same, but SEs are adjusted.

   ```python
   ols_model = sm.OLS(y, X).fit(cov_type='HC3')
   ```

4. **Model Redesign**

   * Add missing variables
   * Use nonlinear models (polynomial regression, tree-based methods, etc.)
   * Feature scaling / transformation of predictors

5. **Generalized Least Squares (GLS)**

   * Explicitly models error structure.
   * Useful if heteroscedasticity pattern is well understood.

---

## **🔹 Interview-Style Summary**

👉 Heteroscedasticity means error variance is unequal.
👉 It does not bias coefficients but makes hypothesis testing unreliable.
👉 Detection: residual plots, BP test, White test.
👉 Treatment: log/Box-Cox transforms, WLS, GLS, or robust standard errors.
👉 In practice, **robust standard errors or log transformation** are the most common fixes.


### ❓ Q2. What is Multicollinearity, and how do you treat it?

**🔹 Definition:**
Multicollinearity occurs when two or more independent (predictor) variables in a regression model are **highly correlated** with each other.

* This makes it difficult for the model to determine the **unique effect** of each predictor on the target variable.
* In extreme cases, it leads to unstable coefficients and inflated standard errors.

---

**🔹 Example:**
Suppose you are predicting `house_price` using `size_in_sqft` and `number_of_rooms`.

* Since larger houses generally have more rooms, these two predictors may be highly correlated, causing multicollinearity.

---

**🔹 Problems caused by Multicollinearity:**

1. Coefficients become **unstable** (small changes in data → large changes in β).
2. Inflated **standard errors** → t-tests may wrongly show predictors as insignificant.
3. Difficulty in interpreting predictor importance.

---

**🔹 Detection Methods:**

* **Correlation Matrix:** High correlation (≥ 0.8 or 0.9) between predictors.
* **Variance Inflation Factor (VIF):**

  * VIF > 5 (sometimes > 10) indicates high multicollinearity.

```python
# Example: Checking VIF in Python
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

X = df[['size_in_sqft', 'num_rooms', 'num_bathrooms']]  # predictors
X = sm.add_constant(X)

vif = pd.DataFrame()
vif["Feature"] = X.columns
vif["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print(vif)
```

---

**🔹 Treatment / Remedies:**

1. **Remove one of the correlated variables** (e.g., drop `num_rooms` if highly correlated with `size_in_sqft`).
2. **Combine variables** using **PCA (Principal Component Analysis)** or feature engineering.
3. **Regularization (Ridge/Lasso Regression):**

   * Ridge shrinks correlated coefficients toward each other.
   * Lasso can drop redundant variables by assigning zero coefficients.
4. **Collect more data** (sometimes multicollinearity is reduced with a larger dataset).
5. **Centering variables (standardization/mean-centering):** Helps reduce correlation in polynomial terms.

---

✅ **Interview Tip:**

> “Multicollinearity doesn’t reduce the predictive power of the model drastically, but it affects interpretability. So, whether to treat it depends on the objective – prediction vs. interpretation.”


## Q3. What is Market Basket Analysis? How would you do it in Python?

### 📌 Theory & Intuition

* **Market Basket Analysis (MBA)** is a technique used in retail and e-commerce to understand the **purchase behavior of customers**.
* It identifies **associations or co-occurrence relationships** between items purchased together.
* Example: If a customer buys *bread*, they are more likely to buy *butter*.

### 🔹 Key Concepts

* **Association Rule Mining**: Finds relationships between items.
* **Support**: Probability of items appearing together in transactions.

  $$
  \text{Support(A → B)} = \frac{\text{Transactions containing (A ∪ B)}}{\text{Total transactions}}
  $$
* **Confidence**: Probability of buying B given A.

  $$
  \text{Confidence(A → B)} = \frac{\text{Support(A ∪ B)}}{\text{Support(A)}}
  $$
* **Lift**: Strength of association relative to independence.

  $$
  \text{Lift(A → B)} = \frac{\text{Confidence(A → B)}}{\text{Support(B)}}
  $$

  * Lift > 1 → Positive association
  * Lift = 1 → Independent
  * Lift < 1 → Negative association

---

## 📌 Python Example (Using `mlxtend`)

```python
# Install mlxtend if not available
# !pip install mlxtend

import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

# Sample dataset: Transactions
dataset = [
    ['milk', 'bread', 'butter'],
    ['bread', 'diapers', 'beer'],
    ['milk', 'bread', 'diapers', 'butter'],
    ['bread', 'butter'],
    ['milk', 'diapers', 'beer', 'cola']
]

# Convert dataset into one-hot encoded DataFrame
from mlxtend.preprocessing import TransactionEncoder

te = TransactionEncoder()
te_data = te.fit(dataset).transform(dataset)
df = pd.DataFrame(te_data, columns=te.columns_)

print("One-Hot Encoded Data:")
print(df.head())

# Step 1: Find frequent itemsets
frequent_itemsets = apriori(df, min_support=0.3, use_colnames=True)
print("\nFrequent Itemsets:")
print(frequent_itemsets)

# Step 2: Generate association rules
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)
print("\nAssociation Rules:")
print(rules[['antecedents','consequents','support','confidence','lift']])
```

---

## 📌 Output Interpretation

* The rules table will show:

  * **Antecedents → Consequents** (e.g., `{milk} → {bread}`)
  * **Support** (how often they occur together)
  * **Confidence** (likelihood of consequent given antecedent)
  * **Lift** (strength of the relationship)

---

## 📌 Use Cases

* **Retail**: Product bundling (e.g., “Buy chips, get soda”).
* **E-commerce**: Recommender systems (“Customers also bought…”).
* **Healthcare**: Drug prescription patterns.
* **Finance**: Fraud detection (suspicious transaction combinations).

---

✅ In interviews, mention both **theory (support, confidence, lift)** and **implementation (Apriori in `mlxtend`)**.




## Q4. What is Association Analysis? Where is it used?

---

### ✅ Definition

* Association Analysis is a **data mining technique** used to discover **relationships, patterns, or associations** between variables/items in large datasets.
* It identifies **if-then rules** (called **association rules**) of the form:

  ```
  IF item A → THEN item B
  ```

  Example: "If a customer buys bread, they are likely to buy butter."

---

### ✅ Key Concepts

* **Support** → Frequency of an itemset in the dataset.
* **Confidence** → Likelihood that item B is bought when item A is bought.
* **Lift** → Strength of association compared to random chance.

---

### ✅ Where is it used?

1. **Market Basket Analysis**

   * Retail stores to identify products often bought together.
   * Example: Amazon "Frequently Bought Together".

2. **Recommender Systems**

   * Suggesting items/movies based on association rules.

3. **Cross-selling / Upselling**

   * Banks recommending credit cards with savings accounts.

4. **Healthcare**

   * Finding correlations between symptoms and diseases.

5. **Fraud Detection**

   * Discover unusual item combinations in financial transactions.

---

### ✅ Python Example (Using Apriori)

```python
from mlxtend.frequent_patterns import apriori, association_rules
import pandas as pd

# Sample dataset
dataset = [
    ['milk', 'bread', 'butter'],
    ['bread', 'butter'],
    ['milk', 'bread'],
    ['milk', 'bread', 'butter', 'jam'],
    ['bread', 'jam']
]

# Convert dataset to one-hot encoded DataFrame
from mlxtend.preprocessing import TransactionEncoder
te = TransactionEncoder()
data = te.fit(dataset).transform(dataset)
df = pd.DataFrame(data, columns=te.columns_)

# Apply Apriori
frequent_itemsets = apriori(df, min_support=0.4, use_colnames=True)

# Generate rules
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1)
print(rules[['antecedents','consequents','support','confidence','lift']])
```

---

### ✅ Interview Tip

👉 Always connect **Association Analysis** with **Market Basket Analysis**, as that’s the most common use case.
👉 Highlight **Support, Confidence, Lift** since interviewers often test your grasp on these metrics.



## ❓ Q5. What is KNN Classifier?

### 🔹 Intuition

The **K-Nearest Neighbors (KNN)** algorithm is a **supervised learning method** used for both **classification and regression**.
It makes predictions based on the **majority class (for classification)** or **average values (for regression)** of the *k closest data points* in the feature space.

It assumes:

> "Similar data points exist close to each other in the feature space."

---

### 🔹 How It Works

1. Choose a value of **k** (number of neighbors).
2. Compute the **distance** (Euclidean, Manhattan, or Minkowski) between the test sample and all training samples.
3. Select the **k nearest neighbors**.
4. Perform:

   * **Classification:** Assign the most frequent class among neighbors.
   * **Regression:** Take the average of neighbors’ values.

---

### 🔹 Advantages

* Simple and intuitive.
* No training phase (lazy learning).
* Works well for small datasets.

### 🔹 Disadvantages

* Computationally expensive for large datasets.
* Sensitive to irrelevant/noisy features and scaling.
* Performance depends on the choice of **k** and distance metric.

---

### 🔹 Applications

* **Recommendation Systems** (e.g., suggesting movies based on similar users).
* **Medical Diagnosis** (classifying diseases based on symptoms).
* **Anomaly Detection** (detecting fraud transactions).

---

### 🔹 Python Example

```python
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# Load dataset
iris = load_iris()
X, y = iris.data, iris.target

# Split into train and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Build KNN model
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train, y_train)

# Predictions
y_pred = knn.predict(X_test)

# Accuracy
print("Accuracy:", accuracy_score(y_test, y_pred))
```

---

✅ **In short:**
KNN is a **distance-based, instance-learning algorithm** that classifies points based on the majority vote of nearest neighbors.

